    # Dataset Reddit của JODIE -> tiền xử lý TGN -> TIGER

Notebook này làm theo README của repository: tải **dataset Reddit chính thức của JODIE** (không scrape comment bằng Reddit API), tiền xử lý theo cách của TGN rồi chạy TIGER cho **temporal link prediction**. Dataset gốc có nhiều hơn 25.000 interactions; notebook dùng toàn bộ dữ liệu thay vì dừng ở 25.000.

> Reddit là website/nền tảng dữ liệu. Redis là một cơ sở dữ liệu cache/key-value riêng và không được README yêu cầu. Notebook này không cần Redis server.

Chạy tuần tự từ thư mục gốc repository. Dataset được tải từ [JODIE project page](https://snap.stanford.edu/jodie/) và các file kết quả được lưu trong `data/` theo schema mà `load_jodie_data` của repository đọc.

In [ ]:
%pip install numpy pandas

In [ ]:
import csv
import subprocess
import sys
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
RAW_PATH = DATA_DIR / "reddit.csv"
PROCESSED_PATH = DATA_DIR / "ml_reddit.csv"
EDGE_FEATURE_PATH = DATA_DIR / "ml_reddit.npy"
NODE_FEATURE_PATH = DATA_DIR / "ml_reddit_node.npy"
DATA_URL = "https://snap.stanford.edu/jodie/reddit.csv"
MIN_RECORDS = 25_000

if not (PROJECT_ROOT / "train_self_supervised.py").is_file():
    raise FileNotFoundError("Hãy mở/chạy notebook từ thư mục gốc www2023tiger.")
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_PATH.is_file() or RAW_PATH.stat().st_size == 0:
    print("Đang tải dataset Reddit chính thức từ JODIE...")
    urllib.request.urlretrieve(DATA_URL, RAW_PATH)
if RAW_PATH.stat().st_size == 0:
    raise RuntimeError(f"Tải dataset thất bại hoặc file rỗng: {RAW_PATH}")

print(f"Dataset nguồn: {RAW_PATH}")
print(f"Kích thước: {RAW_PATH.stat().st_size / (1024 * 1024):.1f} MiB")

## Tiền xử lý theo TGN

CSV JODIE có user ID, item ID, timestamp, state label và vector đặc trưng tương tác. Cell dưới đây áp dụng quy tắc `--bipartite` của TGN: tách miền user/item thành hai miền node không chồng lấn, cộng 1 để ID 0 dành cho padding, cộng 1 cho `idx`, thêm hàng feature zero ở vị trí 0 và tạo node feature zero 172 chiều. Không cắt dữ liệu còn 25.000; kiểm tra đảm bảo dataset thực tế đủ số lượng trước khi train.

In [ ]:
users, items, timestamps, labels, edge_features = [], [], [], [], []
with RAW_PATH.open("r", newline="", encoding="utf-8") as source:
    reader = csv.reader(source)
    header = next(reader, None)
    if not header or len(header) < 5:
        raise ValueError("Header của JODIE Reddit không đúng định dạng mong đợi.")

    for line_number, row in enumerate(reader, start=2):
        if len(row) < 5:
            raise ValueError(f"Dòng {line_number} thiếu cột/features trong {RAW_PATH}.")
        try:
            users.append(int(row[0]))
            items.append(int(row[1]))
            timestamps.append(float(row[2]))
            labels.append(float(row[3]))
            edge_features.append([float(value) for value in row[4:]])
        except ValueError as exc:
            raise ValueError(f"Giá trị không hợp lệ tại dòng {line_number}.") from exc

if len(users) < MIN_RECORDS:
    raise RuntimeError(
        f"Dataset chỉ có {len(users):,} interactions; cần ít nhất {MIN_RECORDS:,}. "
        "Không tiếp tục train với dữ liệu thiếu. Kiểm tra file nguồn/tải lại từ JODIE."
    )
features = np.asarray(edge_features, dtype=np.float32)
if features.ndim != 2 or features.shape[0] != len(users) or not np.isfinite(features).all():
    raise ValueError("Edge feature array không hợp lệ hoặc có kích thước không khớp interactions.")
if not np.isfinite(timestamps).all() or not np.isfinite(labels).all():
    raise ValueError("Timestamp/label có giá trị NaN hoặc vô hạn.")

raw = pd.DataFrame({
    "u": np.asarray(users, dtype=np.int64),
    "i": np.asarray(items, dtype=np.int64),
    "ts": np.asarray(timestamps, dtype=np.float64),
    "label": np.asarray(labels, dtype=np.float32),
    "idx": np.arange(len(users), dtype=np.int64),
})
if not raw["ts"].is_monotonic_increasing:
    raise ValueError("Dataset JODIE không theo thứ tự timestamp; không tự sắp xếp vì phải giữ thứ tự feature/index gốc.")

# TGN's bipartite reindexing keeps user IDs and item IDs in disjoint node ranges.
if raw["u"].max() - raw["u"].min() + 1 != raw["u"].nunique():
    raise ValueError("User IDs không liên tục như yêu cầu của bước reindex bipartite TGN.")
if raw["i"].max() - raw["i"].min() + 1 != raw["i"].nunique():
    raise ValueError("Item IDs không liên tục như yêu cầu của bước reindex bipartite TGN.")
upper_user_id = int(raw["u"].max()) + 1
processed = raw.copy()
processed["i"] = processed["i"] + upper_user_id
processed["u"] = processed["u"] + 1
processed["i"] = processed["i"] + 1
processed["idx"] = processed["idx"] + 1
processed = processed[["u", "i", "ts", "label", "idx"]]

edge_features = np.vstack((np.zeros((1, features.shape[1]), dtype=np.float32), features))
max_node_id = int(max(processed["u"].max(), processed["i"].max()))
node_features = np.zeros((max_node_id + 1, 172), dtype=np.float32)

if len(processed) != len(features) or processed["idx"].iloc[0] != 1:
    raise AssertionError("Interaction/feature alignment check failed.")
if processed["u"].min() <= 0 or processed["i"].min() <= 0:
    raise AssertionError("Node ID 0 must remain reserved for padding.")

processed.to_csv(PROCESSED_PATH, index=False)
np.save(EDGE_FEATURE_PATH, edge_features)
np.save(NODE_FEATURE_PATH, node_features)

check = pd.read_csv(PROCESSED_PATH)
if list(check.columns) != ["u", "i", "ts", "label", "idx"] or len(check) < MIN_RECORDS:
    raise RuntimeError("Kiểm tra output TIGER thất bại: sai schema hoặc thiếu interactions.")
if np.load(EDGE_FEATURE_PATH, mmap_mode="r").shape != (len(check) + 1, features.shape[1]):
    raise RuntimeError("Edge feature file không khớp số interactions.")
if np.load(NODE_FEATURE_PATH, mmap_mode="r").shape != (max_node_id + 1, 172):
    raise RuntimeError("Node feature file sai kích thước.")

print(f"Số interactions: {len(check):,}")
print(f"Số nodes: {max_node_id:,}; edge feature dimension: {features.shape[1]}")
print(f"Đã tạo: {PROCESSED_PATH.name}, {EDGE_FEATURE_PATH.name}, {NODE_FEATURE_PATH.name}")

## Chạy TIGER temporal link prediction

Dataset Reddit đã có edge features 172 chiều nên để model tự suy ra chiều mặc định (không ép `--dim 64`). `restart_prob=0` cho phép chạy baseline không restart; dùng `static` restarter để khớp cấu hình không restart. Có thể tăng số epoch sau khi xác nhận lần chạy ngắn hoạt động.

In [ ]:
N_EPOCHS = 5
BATCH_SIZE = 200
import torch
gpu = "0" if torch.cuda.is_available() else "-1"

command = [
    sys.executable, "train_self_supervised.py",
    "--data", "reddit",
    "--root", str(PROJECT_ROOT),
    "--gpu", gpu,
    "--n_epochs", str(N_EPOCHS),
    "--bs", str(BATCH_SIZE),
    "--restart_prob", "0",
    "--restarter_type", "static",
]
print(f"Training on {'GPU ' + gpu if gpu != '-1' else 'CPU'} with {len(check):,} interactions...")
subprocess.run(command, cwd=PROJECT_ROOT, check=True)
print("Huấn luyện hoàn tất; xem thư mục results/ để lấy metric và saved_models/ để lấy model.")